In [1]:
import csv
import pandas as pd
import scipy.io
import numpy as np
import copy

import matplotlib.pyplot as plt
import h5py
import scipy
from PIL import Image
from scipy import ndimage


from tensorflow import keras
import tensorflow as tf

import sklearn

import os
from tensorflow.keras.callbacks import CSVLogger, ModelCheckpoint
from tensorflow.keras.models import load_model


%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install watermark
%load_ext watermark

%watermark --iversions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.6 MB/s eta 0:00:00
keras     : 3.8.0
scipy     : 1.15.3
tensorflow: 2.18.0
numpy     : 2.0.2
google    : 2.0.3
PIL       : 11.2.1
pandas    : 2.2.2
h5py      : 3.14.0
sklearn   : 1.6.1
csv       : 1.0
matplotlib: 3.10.0



In [4]:
bal_all = np.loadtxt('/content/drive/MyDrive/ML/Disease_classification/Dataset/both_phases.csv', delimiter=',')
x_bal = bal_all[:,0:1024]
y_disease = bal_all[:,-1]
print(np.shape(x_bal))
print(np.shape(y_disease))
y_disease=y_disease.reshape(-1,1 )
print(np.shape(y_disease))

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

print(y_disease[8500,0])
enc = OneHotEncoder(handle_unknown='ignore')
# 2. FIT
enc.fit(y_disease)

# 3. Transform
onehotlabels = enc.transform(y_disease).toarray()

print(onehotlabels[8500])

x_train1, x_test1, y_train1, y_test1 = train_test_split(x_bal, onehotlabels, test_size = 0.2, shuffle = True)

(29310, 1024)
(29310,)
(29310, 1)
1.0
[0. 1. 0. 0. 0.]


In [8]:
from drive.MyDrive.ML.Disease_classification.EEGPT.downstream.Modules.models.EEGPT_mcae_finetune import *
import torch
import numpy as np

encoder = EEGPTClassifier.from_pretrained("/content/drive/MyDrive/ML/Disease_classification/EEGPT/checkpoint/eegpt_mcae_58chs_4s_large4E.ckpt")

X = torch.tensor(x_train1, dtype=torch.float32)
y = torch.tensor(np.argmax(y_train1, 1), dtype=torch.long)

# Step 4: Fine-tune model
encoder.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

# Step 5: Evaluate
accuracy = encoder.evaluate(torch.tensor(x_test1, dtype=torch.float32),
                            torch.tensor(np.argmax(y_test1, 1), dtype=torch.long))
print("EEGPT Test Accuracy: {:.2f}%".format(accuracy * 100))


AttributeError: type object 'EEGPTClassifier' has no attribute 'from_pretrained'